# ViFinQA — Qwen2.5-Coder-7B Select, Payload w=0.10 Countfix, Codegen k=5

So với bản `w010 k=5` đạt `EXEC 0.2016`, notebook này giữ `--k 5` và chỉ đổi payload/code retrieval mới:

- route count lấy metric từ điều kiện cần đếm thay vì `co/so/cac`.
- selection hỗ trợ op `count` để trả số lượng có dataframe-grounded pandas query.

Dataset payload cần attach/upload:

```text
/kaggle/input/datasets/kien2005/kaggle-payload-w010-countfix
```

Output cần tải về:

```text
/kaggle/working/codegen_sel7b_w010_countfix_k5.jsonl
```


In [ ]:
import glob, json, pathlib

EXPLICIT_ROOT = pathlib.Path("/kaggle/input/datasets/kien2005/kaggle-payload-w010-countfix")
hits = []
if EXPLICIT_ROOT.exists():
    hits = sorted(EXPLICIT_ROOT.glob("**/retrieval.jsonl"))

if not hits:
    all_hits = [pathlib.Path(p) for p in glob.glob("/kaggle/input/**/retrieval.jsonl", recursive=True)]
    hits = [p for p in all_hits if "countfix" in str(p).lower() or "w010-count" in str(p).lower()]
    if not hits:
        hits = all_hits

assert hits, "Chưa attach dataset payload w010-countfix hoặc Kaggle chưa mount input."
assert len(hits) == 1, f"Có nhiều retrieval.jsonl; hãy detach payload cũ hoặc chỉ rõ path: {hits}"

PAYLOAD = str(hits[0].parent)
manifest_path = pathlib.Path(PAYLOAD) / "payload-manifest.json"
assert manifest_path.exists(), f"Payload thiếu manifest: {manifest_path}"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
assert manifest.get("schema_version") == 2, f"Payload schema cũ: {manifest.get('schema_version')}"

print("PAYLOAD =", PAYLOAD)
print("manifest files =", len(manifest.get("files", {})))

import torch
print("GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU 0:", torch.cuda.get_device_name(0))


In [ ]:
import pathlib, shutil

SRC = pathlib.Path(PAYLOAD) / "code"
DST = pathlib.Path("/kaggle/working/code")
assert SRC.exists(), f"Payload thieu code/: {SRC}"
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
print("code ->", DST)


In [ ]:
%%time
!pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"
import transformers, bitsandbytes
print("transformers", transformers.__version__)
print("bitsandbytes", bitsandbytes.__version__)


In [ ]:
%%time
# Smoke test: same config, limit=12.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-7B-Instruct --load-4bit \
    --llm-mode select --llm-target all \
    --out /kaggle/working/codegen_smoke_w010_countfix_k5.jsonl --limit 12 \
    --n 1 --temperature 0 --k 5 --max-tokens 96 --batch-size 4 \
    --checkpoint-every 4 --time-budget-min 30 --seed 13


In [ ]:
import collections, json, pathlib

smoke = pathlib.Path("/kaggle/working/codegen_smoke_w010_countfix_k5.jsonl")
rows = [json.loads(line) for line in smoke.open(encoding="utf-8")]
print("rows", len(rows), "unique ids", len({r["id"] for r in rows}))
print(collections.Counter(r.get("source") for r in rows))
print(rows[-1])


In [ ]:
%%time
# Full run. Chạy lại cùng cell + cùng output/config sẽ resume theo run_signature.
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-7B-Instruct --load-4bit \
    --llm-mode select --llm-target all \
    --out /kaggle/working/codegen_sel7b_w010_countfix_k5.jsonl \
    --n 1 --k 5 --max-tokens 96 --batch-size 8 \
    --checkpoint-every 32 --time-budget-min 400 --seed 13


In [ ]:
import collections, json, math, pathlib

out = pathlib.Path("/kaggle/working/codegen_sel7b_w010_countfix_k5.jsonl")
rows = [json.loads(line) for line in out.open(encoding="utf-8")]
ids = [r["id"] for r in rows]
assert len(rows) == 1012 and len(set(ids)) == 1012, (len(rows), len(set(ids)))
assert all(math.isfinite(float(r.get("answer", 0.0))) for r in rows)
print(collections.Counter(r.get("source") for r in rows))
print("OK: 1012 unique finite results ->", out)


## Sau Khi Chạy Xong

Tải file này về local:

```text
/kaggle/working/codegen_sel7b_w010_countfix_k5.jsonl
```

Build submission local:

```bash
python scripts/05_build_submission.py \
  --retrieval artifacts/retrieval_p1_rowrerank_full_w010_count.jsonl \
  --codegen <duong_dan>/codegen_sel7b_w010_countfix_k5.jsonl \
  --out-dir artifacts/submission_sel7b_w010_countfix_codegen_k5
```
